# Creating a TF-IDF baseline

## Prepare Input
We use the approved 768/512 chunking settings (chunk length / chunk overlap) to establish a TF-IDF baseline. 



In [ ]:
import hashlib
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
GATE3_DIR = PROJECT_ROOT / "data" / "gate3"

def load_chunks(path):
    '''Loads chunks from CSV files while mainta'''
    df = pd.read_csv(path)
    # normalize line endings on chunk files to account for Git rewriting LF -> CRLF
    # TODO: remove manual replacement once .gitattributes is in main and local copies get refreshed 
    df["text"] = df["text"].str.replace("\r\n", "\n", regex=False)
    hashes = df["text"].map(lambda t: hashlib.sha256(t.encode("utf-8")).hexdigest())
    bad = (hashes != df["text_hash"]).sum()
    assert bad == 0, f"{bad} chunks in {path.name} fail the text_hash check"
    return df

train_chunks = load_chunks(GATE3_DIR / "train_chunks_768_512.csv")
val_chunks = load_chunks(GATE3_DIR / "validation_chunks_768_512.csv")


## TF-IDF Fixed Settings

Use one shared TF-IDF vectorizer fitted only on model-training chunk text. Validation and official-test text may use the fitted vectorizer but must not change its vocabulary or weights. When training each category's classifier, exclude the rows masked for that category.

Use these fixed vectorizer settings:

- normalized text produced with the same four steps used by the keyword baseline;
- word tokens matched by `(?u)\b\w+\b`, including one-character and numeric tokens;
- single words and two-word phrases: `ngram_range=(1, 2)`;
- `min_df` is the candidate value `2` or `5`;
- `use_idf=True`;
- `smooth_idf=True`;
- `sublinear_tf=True`;
- L2 row normalization;
- `max_df=1.0`; and
- no maximum feature limit.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 2), use_idf=True, smooth_idf=True, sublinear_tf=True, 
                             norm='l2', max_df=1.0, max_features=None, stop_words=None,)

## Normalize text

1. Unicode NFKC normalization
2. Unicode case-folding
3. Consistent equivalents for curly quotes and dashes
4. Collapse each run of whitespace to one space



In [11]:
# importing from a script, since multiple notebooks will need the same steps
from text_normalization import normalize_text

train_chunks["norm_text"] = train_chunks["text"].map(normalize_text)
val_chunks["norm_text"] = val_chunks["text"].map(normalize_text)


## Evaluating 6 Configurations
using every combination of:
- `min_df`: 2 or 5; and
- `C`: 0.1, 1.0, or 10.0.